In [4]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv
/kaggle/input/datasets/lukajincharadze/pandas-data/noc_regions.csv
/kaggle/input/datasets/lukajincharadze/pandas-data/olympics-data.xlsx
/kaggle/input/datasets/lukajincharadze/pandas-data/results.parquet
/kaggle/input/datasets/lukajincharadze/pandas-data/bios.csv
/kaggle/input/datasets/lukajincharadze/pandas-data/coffee.csv
/kaggle/input/datasets/lukajincharadze/pandas-data/results.csv


In [5]:
%pip install -q dagshub mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 857.6 kB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 91.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 78.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 69.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━

# DagsHub/Mlflow Initialization

In [6]:
import dagshub
dagshub.init(repo_owner='skoba23', repo_name='ML_Assignment_2', mlflow=True)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=19e0ad0f-9af9-4d55-be91-589b7c5d6b4a&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=98907d1b11db3dc570b7a87af95aac73e96023f2cc6eb4a866123c1c3621ebab




Output()

Accessing as skoba23

Initialized MLflow to track repo "skoba23/ML_Assignment_2"

Repository skoba23/ML_Assignment_2 initialized!

In [7]:
import mlflow
import mlflow.sklearn
import os
TRACKING_URI = "https://dagshub.com/skoba23/ML_Assignment_2.mlflow"

mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment("XGBoost_Training")

print('MLflow Tracking URI:', TRACKING_URI)
print('Setup complete!')

MLflow Tracking URI: https://dagshub.com/skoba23/ML_Assignment_2.mlflow
Setup complete!


## Data Loading

In [8]:
train_transaction = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')
train_identity    = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv')
test_transaction  = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv')
test_identity     = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv')

train = train_transaction.merge(train_identity, on='TransactionID', how='left')
test  = test_transaction.merge(test_identity,   on='TransactionID', how='left')

print('Train shape:', train.shape)
print('Test shape: ', test.shape)
print('Fraud rate: ', round(train['isFraud'].mean() * 100, 2), '%')

Train shape: (590540, 434)
Test shape:  (506691, 433)
Fraud rate:  3.5 %


# Cleaning

In [9]:
mlflow.start_run(run_name="XGB_Cleaning")

null_threshold = 0.7
null_ratio = train.isnull().mean()
drop_cols = null_ratio[null_ratio > null_threshold].index.tolist()
drop_cols = [c for c in drop_cols if c != "isFraud"]

df_train_clean = train.drop(columns=drop_cols)
df_test_clean  = test.drop(columns=[c for c in drop_cols if c in test.columns])

train = train.sample(frac=0.3, random_state=42)
print(train.shape)

(177162, 434)


### Filling Null Values With Median

In [10]:
num_cols = df_train_clean.select_dtypes(include=np.number).columns.tolist()
num_cols = [c for c in num_cols if c != "isFraud"]
df_train_clean[num_cols] = df_train_clean[num_cols].fillna(df_train_clean[num_cols].median())
df_test_clean[num_cols]  = df_test_clean[[c for c in num_cols if c in df_test_clean.columns]].fillna(
                            df_train_clean[[c for c in num_cols if c in df_test_clean.columns]].median())

### One-Hot Encoding

In [11]:
from sklearn.preprocessing import LabelEncoder
cat_cols = df_train_clean.select_dtypes(include="object").columns.tolist()
for col in cat_cols:
    le = LabelEncoder()
    df_train_clean[col] = le.fit_transform(df_train_clean[col].astype(str))
    if col in df_test_clean.columns:
        df_test_clean[col] = le.transform(df_test_clean[col].astype(str).map(
            lambda x: x if x in le.classes_ else le.classes_[0]))

In [12]:
mlflow.log_param("null_threshold", null_threshold)
mlflow.log_param("encoding", "LabelEncoder")
mlflow.log_param("dropped_columns", len(drop_cols))
mlflow.log_metric("remaining_features", df_train_clean.shape[1])
print(f"Dropped {len(drop_cols)} columns, remaining: {df_train_clean.shape[1]}")

Dropped 208 columns, remaining: 226


In [13]:
mlflow.end_run()

🏃 View run XGB_Cleaning at: https://dagshub.com/skoba23/ML_Assignment_2.mlflow/#/experiments/3/runs/4ac16df96ec548b99381ec2462cbe55f
🧪 View experiment at: https://dagshub.com/skoba23/ML_Assignment_2.mlflow/#/experiments/3


# Feature Engineering

In [14]:
with mlflow.start_run(run_name="XGB_FeatureEngineering"):
    df_train_clean["TransactionAmt_log"] = np.log1p(df_train_clean["TransactionAmt"])
    df_test_clean["TransactionAmt_log"]  = np.log1p(df_test_clean["TransactionAmt"])

    df_train_clean["Transaction_day"]  = (df_train_clean["TransactionDT"] // (3600 * 24)) % 7
    df_train_clean["Transaction_hour"] = (df_train_clean["TransactionDT"] // 3600) % 24
    df_test_clean["Transaction_day"]   = (df_test_clean["TransactionDT"] // (3600 * 24)) % 7
    df_test_clean["Transaction_hour"]  = (df_test_clean["TransactionDT"] // 3600) % 24

    df_train_clean["card1_addr1"] = (df_train_clean["card1"].astype(str) + "_" + df_train_clean["addr1"].astype(str))
    df_test_clean["card1_addr1"]  = (df_test_clean["card1"].astype(str) + "_" + df_test_clean["addr1"].astype(str))

    le = LabelEncoder()
    df_train_clean["card1_addr1"] = le.fit_transform(df_train_clean["card1_addr1"].astype(str))

    label_map = {cls: idx for idx, cls in enumerate(le.classes_)}
    df_test_clean["card1_addr1"] = (
        df_test_clean["card1_addr1"].astype(str).map(label_map).fillna(0).astype(int)
    )

    fraud_rate = df_train_clean.groupby("card1")["isFraud"].mean()
    df_train_clean["card1_fraud_rate"] = df_train_clean["card1"].map(fraud_rate)
    df_test_clean["card1_fraud_rate"]  = df_test_clean["card1"].map(fraud_rate).fillna(fraud_rate.mean())

    mean_amt = df_train_clean["TransactionAmt"].mean()
    std_amt  = df_train_clean["TransactionAmt"].std()
    df_train_clean["TransactionAmt_zscore"] = (df_train_clean["TransactionAmt"] - mean_amt) / std_amt
    df_test_clean["TransactionAmt_zscore"]  = (df_test_clean["TransactionAmt"] - mean_amt) / std_amt

    mlflow.log_param("new_features", "TransactionAmt_log, Transaction_day, Transaction_hour, card1_addr1, card1_fraud_rate, TransactionAmt_zscore")
    mlflow.log_metric("total_features", df_train_clean.shape[1])
    print("Feature Engineering done:", df_train_clean.shape)

/tmp/ipykernel_57/3819496944.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_train_clean["TransactionAmt_log"] = np.log1p(df_train_clean["TransactionAmt"])
/tmp/ipykernel_57/3819496944.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_test_clean["TransactionAmt_log"]  = np.log1p(df_test_clean["TransactionAmt"])
/tmp/ipykernel_57/3819496944.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all column

Feature Engineering done: (590540, 232)
🏃 View run XGB_FeatureEngineering at: https://dagshub.com/skoba23/ML_Assignment_2.mlflow/#/experiments/3/runs/e8578baeff5d4e989e0db8e290afcaf9
🧪 View experiment at: https://dagshub.com/skoba23/ML_Assignment_2.mlflow/#/experiments/3


# Feature Selection

In [15]:
from sklearn.feature_selection import SelectKBest, f_classif

Y = df_train_clean["isFraud"]
X = df_train_clean.drop(columns=["isFraud", "TransactionID"])

for col in X.select_dtypes(include="object").columns:
    X[col] = X[col].astype("category").cat.codes

mlflow.start_run(run_name="XGB_FeatureSelection")

<ActiveRun: >

In [16]:
corr = X.corrwith(Y).abs()
selected_features = corr[corr > 0.01].index.tolist()

mlflow.log_param("method", "correlation_threshold")
mlflow.log_param("threshold", 0.01)
mlflow.log_metric("selected_features", len(selected_features))
print(f"Selected {len(selected_features)} features")
mlflow.end_run()

X_final = X[selected_features]
X_test_final = df_test_clean[[c for c in selected_features if c in df_test_clean.columns]]

Selected 165 features
🏃 View run XGB_FeatureSelection at: https://dagshub.com/skoba23/ML_Assignment_2.mlflow/#/experiments/3/runs/42155464f9e1461b856fd4127e059339
🧪 View experiment at: https://dagshub.com/skoba23/ML_Assignment_2.mlflow/#/experiments/3


### Filling missing columns with 0

In [17]:
for col in selected_features:
    if col not in X_test_final.columns:
        X_test_final[col] = 0
X_test_final = X_test_final[selected_features]

# Training

In [18]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

X_tr, X_val, Y_tr, Y_val = train_test_split(X_final, Y, test_size=0.2, random_state=42, stratify=Y)

scale_pos_weight = (Y_tr == 0).sum() / (Y_tr == 1).sum()
print(f"scale_pos_weight: {scale_pos_weight:.2f}")

scale_pos_weight: 27.58


### Baseline

In [19]:
from xgboost import XGBClassifier
mlflow.start_run(run_name="XGB_Train_Baseline")

pipe1 = Pipeline([
    ("model", XGBClassifier(n_estimators=100, random_state=42,
                             eval_metric="auc", verbosity=0))
])
pipe1.fit(X_tr, Y_tr)
preds1 = pipe1.predict_proba(X_val)[:, 1]
auc1 = roc_auc_score(Y_val, preds1)

mlflow.log_params({"n_estimators": 100, "max_depth": 6, "learning_rate": 0.3})
mlflow.log_metric("val_auc", auc1)
mlflow.sklearn.log_model(pipe1, "model")
print(f"Baseline AUC: {auc1:.4f}")
mlflow.end_run()

2026/05/08 13:13:58 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/08 13:13:59 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Baseline AUC: 0.9582
🏃 View run XGB_Train_Baseline at: https://dagshub.com/skoba23/ML_Assignment_2.mlflow/#/experiments/3/runs/fcb6645abd9840289e57dcda7fc5ae88
🧪 View experiment at: https://dagshub.com/skoba23/ML_Assignment_2.mlflow/#/experiments/3


### Decreasing Learning Rate

In [20]:
mlflow.start_run(run_name="XGB_Train_LearningRateDec")

pipe2 = Pipeline([
    ("model", XGBClassifier(n_estimators=300, learning_rate=0.01,
                             max_depth=6, random_state=42,
                             eval_metric="auc", verbosity=0))
])
pipe2.fit(X_tr, Y_tr)
preds2 = pipe2.predict_proba(X_val)[:, 1]
auc2 = roc_auc_score(Y_val, preds2)

mlflow.log_params({"n_estimators": 300, "learning_rate": 0.01, "max_depth": 6})
mlflow.log_metric("val_auc", auc2)
mlflow.sklearn.log_model(pipe2, "model")
print(f"LR=0.01 AUC: {auc2:.4f}")
mlflow.end_run()

2026/05/08 13:15:49 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/08 13:15:51 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


LR=0.01 AUC: 0.9338
🏃 View run XGB_Train_LearningRateDec at: https://dagshub.com/skoba23/ML_Assignment_2.mlflow/#/experiments/3/runs/ccd6ba8436c84c558b54fdcc151e53b4
🧪 View experiment at: https://dagshub.com/skoba23/ML_Assignment_2.mlflow/#/experiments/3


### Scale_pos_Weight

In [21]:
mlflow.start_run(run_name="XGB_Train_ScalePosWeight_Balanced")

pipe3 = Pipeline([
    ("model", XGBClassifier(n_estimators=300, learning_rate=0.05,
                             max_depth=6, scale_pos_weight=scale_pos_weight,
                             random_state=42, eval_metric="auc", verbosity=0))
])
pipe3.fit(X_tr, Y_tr)
preds3 = pipe3.predict_proba(X_val)[:, 1]
auc3 = roc_auc_score(Y_val, preds3)

mlflow.log_params({"n_estimators": 300, "learning_rate": 0.05,
                   "max_depth": 6, "scale_pos_weight": round(scale_pos_weight, 2)})
mlflow.log_metric("val_auc", auc3)
mlflow.sklearn.log_model(pipe3, "model")
print(f"ScalePosWeight AUC: {auc3:.4f}")
mlflow.end_run()

2026/05/08 13:16:21 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/08 13:16:21 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


ScalePosWeight AUC: 0.9525
🏃 View run XGB_Train_ScalePosWeight_Balanced at: https://dagshub.com/skoba23/ML_Assignment_2.mlflow/#/experiments/3/runs/f82fe7b99ff04f97b08df2a5e1c849b9
🧪 View experiment at: https://dagshub.com/skoba23/ML_Assignment_2.mlflow/#/experiments/3


### Handling Overfitting

In [22]:
mlflow.start_run(run_name="XGB_Train_Subsample")

pipe4 = Pipeline([
    ("model", XGBClassifier(n_estimators=500, learning_rate=0.05,
                             max_depth=6, subsample=0.8,
                             colsample_bytree=0.8,
                             scale_pos_weight=scale_pos_weight,
                             random_state=42, eval_metric="auc", verbosity=0))
])
pipe4.fit(X_tr, Y_tr)
preds4 = pipe4.predict_proba(X_val)[:, 1]
auc4 = roc_auc_score(Y_val, preds4)

mlflow.log_params({"n_estimators": 500, "learning_rate": 0.05, "max_depth": 6,
                   "subsample": 0.8, "colsample_bytree": 0.8})
mlflow.log_metric("val_auc", auc4)
mlflow.sklearn.log_model(pipe4, "model")
print(f"Subsample AUC: {auc4:.4f}")
mlflow.end_run()

2026/05/08 13:19:20 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/08 13:19:20 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Subsample AUC: 0.9623
🏃 View run XGB_Train_Subsample at: https://dagshub.com/skoba23/ML_Assignment_2.mlflow/#/experiments/3/runs/5f6638703a7149f5997835806baf9f7f
🧪 View experiment at: https://dagshub.com/skoba23/ML_Assignment_2.mlflow/#/experiments/3


### Cross Validation

In [23]:
best_pipe = max([(auc1, pipe1), (auc2, pipe2), (auc3, pipe3), (auc4, pipe4)],
                 key=lambda x: x[0])[1]
best_auc  = max(auc1, auc2, auc3, auc4)

mlflow.start_run(run_name="XGB_CrossValidation")
X_cv_sample, _, Y_cv_sample, _ = train_test_split(X_final, Y, train_size=0.3, random_state=42, stratify=Y)
cv_scores = cross_val_score(best_pipe, X_final, Y, cv=3, scoring="roc_auc")

mlflow.log_param("cv_folds", 3)
mlflow.log_param("cv_sample_size", len(X_cv_sample))
mlflow.log_metric("cv_auc_mean", cv_scores.mean())
mlflow.log_metric("cv_auc_std", cv_scores.std())
print(f"CV AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
mlflow.end_run()

CV AUC: 0.9047 ± 0.0130
🏃 View run XGB_CrossValidation at: https://dagshub.com/skoba23/ML_Assignment_2.mlflow/#/experiments/3/runs/51738f55ea6d4e01842caae67bef228f
🧪 View experiment at: https://dagshub.com/skoba23/ML_Assignment_2.mlflow/#/experiments/3


# Saving Model Registry

In [24]:
mlflow.start_run(run_name="XGB_BestModel_Registry")
mlflow.sklearn.log_model(
    best_pipe,
    artifact_path="model",
    registered_model_name="IEEE_Fraud_XGBoost"
)
mlflow.log_metric("val_auc", best_auc)
print(f"Best model saved! AUC: {best_auc:.4f}")
mlflow.end_run()

2026/05/08 13:21:04 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/08 13:21:04 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'IEEE_Fraud_XGBoost'.
2026/05/08 13:21:12 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: IEEE_Fraud_XGBoost, version 1
Created version '1' of model 'IEEE_Fraud_XGBoost'.


Best model saved! AUC: 0.9623
🏃 View run XGB_BestModel_Registry at: https://dagshub.com/skoba23/ML_Assignment_2.mlflow/#/experiments/3/runs/02d83b33f6894e179b47c023efcc7f0b
🧪 View experiment at: https://dagshub.com/skoba23/ML_Assignment_2.mlflow/#/experiments/3
